# IF/Task Importance Config-Driven Layer-wise Top-p Sparse Update (2D Only)

This notebook creates sparse task-vector merge checkpoints using **layer-wise importance-score top-p masks on 2D parameters only**.

Core behavior:

- Resolve importance artifact path from config pattern:
  - `importance_{task_name}_{mode}_{tail}{top_p}.pt`
  - Example: `importance_if_absolute_top15.pt`
- Load base/task checkpoints and the resolved importance score dictionary.
- **1D parameters** (normalization layers, biases) are **never sparsified** — they always receive the full task-vector delta.
- **2D+ parameters** (attention/MLP weight matrices) are sparsified **per-layer** using exact top-k selection.
- For each sparse keep percentage in `{0.1, 0.5, 1, 5, 10, 20, 50}%`, apply:
  - 2D params: `theta_sparse = theta_base + mask_topk_layerwise(importance) * (theta_task - theta_base)`
  - 1D params: `theta_sparse = theta_task`  (full delta, no masking)
- Save one merged checkpoint + metadata per keep percentage.

Selection mode uses **exact layer-wise top-k** (`torch.topk` per parameter) so the realized kept coordinate count per layer exactly matches the requested ratio. Ties are broken deterministically by flattened index position.

In [2]:
from __future__ import annotations

import gc
import json
import random
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Mapping, Tuple

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer


# --------------------------------------------------------------------------------------
# Runtime configuration
# --------------------------------------------------------------------------------------


DEFAULT_TASK_MODEL_PATHS: Dict[str, Path] = {
    'if': Path('/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-ifrl_ifeval/global_step_50/actor/huggingface'),
    'math': Path('/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-math/stage2/global_step_40/actor/huggingface'),
}


@dataclass(frozen=True)
class RuntimeConfig:
    """Runtime configuration for configurable importance sparse updates.

    Args:
        base_model_id: Base model id/path used as sparse-update anchor.
        task_name: Task key whose task-vector checkpoint is used (`if`, `math`, ...).
        task_model_path_override: Optional explicit task checkpoint path override.
            If `None`, the notebook uses `DEFAULT_TASK_MODEL_PATHS[task_name]`.
        importance_root: Directory containing precomputed importance `.pt` files.
        importance_mode: Importance selection family (`absolute`, `positive`, `negative`).
        importance_tail: Tail token in filename (`top`, `bottom`, or `auto`).
            - `auto` maps to default tail per mode (`absolute/top`, `positive/top`, `negative/bottom`).
        importance_top_p_percent: Source top-p percentage used in importance filename.
            Example: `15.0` -> `...top15.pt`.
        prefer_exact_importance: Prefer `_exact.pt` artifact first when available.
        allow_exact_fallback: Allow fallback to `_exact.pt` when base filename is missing.
        output_root: Root directory where sparse checkpoints and metadata are saved.
        model_dtype_name: Model loading dtype alias (`bf16`, `fp16`, or `fp32`).
        seed: Random seed for deterministic tie handling.
        overwrite_existing: Whether to overwrite existing output checkpoint directories.
    """

    base_model_id: str = 'Qwen/Qwen3-1.7B'
    task_name: str = 'if'
    task_model_path_override: Path | None = None

    importance_root: Path = Path(
        '/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-jwcm-v2-importance-only/importance'
    )
    importance_mode: str = 'positive'
    importance_tail: str = 'auto'
    importance_top_p_percent: float = 15.0
    prefer_exact_importance: bool = False
    allow_exact_fallback: bool = True

    output_root: Path = Path(
        '/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/if_sparse_topk_importance_configurable'
    )
    model_dtype_name: str = 'fp32'
    seed: int = 42
    overwrite_existing: bool = False


RUNTIME = RuntimeConfig()

# Requested sparse keep percentages (importance-score top-p) for mask creation.
SPARSE_KEEP_PERCENTS: Tuple[float, ...] = (0.1, 0.5, 1.0, 5.0, 10.0, 20.0, 50.0)

print(f'Base model: {RUNTIME.base_model_id}')
print(f'Task name: {RUNTIME.task_name}')
print(f'Importance root: {RUNTIME.importance_root}')
print(f'Importance mode/tail/top_p: {RUNTIME.importance_mode}/{RUNTIME.importance_tail}/{RUNTIME.importance_top_p_percent}')
print(f'Output root: {RUNTIME.output_root}')
print(f'Sparse keep percents: {SPARSE_KEEP_PERCENTS}')


Base model: Qwen/Qwen3-1.7B
Task name: if
Importance root: /mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-jwcm-v2-importance-only/importance
Importance mode/tail/top_p: positive/auto/15.0
Output root: /mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/if_sparse_topk_importance_configurable
Sparse keep percents: (0.1, 0.5, 1.0, 5.0, 10.0, 20.0, 50.0)


In [ ]:
# --------------------------------------------------------------------------------------
# Utility and merge helper functions
# --------------------------------------------------------------------------------------


def now_iso() -> str:
    """Return current UTC timestamp in ISO-8601 format.

    Returns:
        UTC timestamp string with second precision.
    """

    return datetime.utcnow().isoformat(timespec='seconds') + 'Z'


def set_seed(seed: int) -> None:
    """Set Python/NumPy/Torch RNG states for reproducible execution.

    Args:
        seed: Integer random seed.

    Returns:
        None. Global RNG states are updated in-place.
    """

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def to_json_compatible(obj: Any) -> Any:
    """Recursively convert runtime objects into JSON-safe values.

    Args:
        obj: Arbitrary runtime object possibly containing `Path` or torch types.

    Returns:
        JSON-serializable object.
    """

    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, torch.dtype):
        return str(obj)
    if isinstance(obj, dict):
        return {str(key): to_json_compatible(value) for key, value in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [to_json_compatible(value) for value in obj]
    return obj


def save_json(payload: Mapping[str, Any], output_path: Path) -> None:
    """Save mapping payload as UTF-8 JSON file.

    Args:
        payload: JSON-compatible mapping payload.
        output_path: Destination path.

    Returns:
        None. File is written to disk.
    """

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open('w', encoding='utf-8') as file:
        json.dump(to_json_compatible(dict(payload)), file, ensure_ascii=False, indent=2)


def resolve_torch_dtype(dtype_name: str) -> torch.dtype:
    """Resolve user-friendly dtype alias into torch dtype object.

    Args:
        dtype_name: Dtype alias string (`bf16`, `fp16`, `fp32`).

    Returns:
        Corresponding torch dtype.

    Raises:
        ValueError: If `dtype_name` is unsupported.
    """

    normalized = dtype_name.strip().lower()
    mapping = {
        'bf16': torch.bfloat16,
        'fp16': torch.float16,
        'fp32': torch.float32,
    }
    if normalized not in mapping:
        raise ValueError(f'Unsupported dtype alias: {dtype_name}')
    return mapping[normalized]


def format_percent_label(percent: float) -> str:
    """Format percentage value into filename-safe label.

    Examples:
        - `15.0` -> `"15"`
        - `12.5` -> `"12_5"`

    Args:
        percent: Percentage value in `(0, 100]`.

    Returns:
        Compact filename-safe string.
    """

    pct = float(percent)
    rounded = round(pct)
    if abs(pct - rounded) < 1e-8:
        return str(int(rounded))
    return f'{pct:.6f}'.rstrip('0').rstrip('.').replace('.', '_')


def keep_percent_to_tag(keep_percent: float) -> str:
    """Convert keep percentage to stable output tag.

    Args:
        keep_percent: Keep percentage in `[0, 100]`.

    Returns:
        Tag like `top_0p1pct` or `top_20pct`.
    """

    pct_str = f'{float(keep_percent):.3f}'.rstrip('0').rstrip('.')
    return f"top_{pct_str.replace('.', 'p')}pct"


def resolve_importance_tail(mode: str, tail: str) -> str:
    """Resolve effective filename tail (`top`/`bottom`) for importance artifacts.

    Args:
        mode: Importance mode (`absolute`, `positive`, `negative`).
        tail: Explicit tail (`top`, `bottom`) or `auto`.

    Returns:
        Resolved tail string (`top` or `bottom`).

    Raises:
        ValueError: If mode/tail is unsupported.
    """

    normalized_mode = mode.strip().lower()
    normalized_tail = tail.strip().lower()

    if normalized_mode not in {'absolute', 'positive', 'negative'}:
        raise ValueError(f'Unsupported importance_mode: {mode}')

    if normalized_tail == 'auto':
        return {
            'absolute': 'top',
            'positive': 'top',
            'negative': 'bottom',
        }[normalized_mode]

    if normalized_tail not in {'top', 'bottom'}:
        raise ValueError(f'Unsupported importance_tail: {tail}')

    return normalized_tail


def resolve_task_model_path(task_name: str, task_model_path_override: Path | None) -> Path:
    """Resolve task model checkpoint path from task key and optional override.

    Args:
        task_name: Task key such as `if` or `math`.
        task_model_path_override: Optional explicit checkpoint path.

    Returns:
        Resolved task model checkpoint path.

    Raises:
        KeyError: If `task_name` is unknown and no override is provided.
    """

    if task_model_path_override is not None:
        return Path(task_model_path_override)

    normalized_task = task_name.strip().lower()
    if normalized_task not in DEFAULT_TASK_MODEL_PATHS:
        raise KeyError(
            f'Unknown task_name={task_name}. Provide task_model_path_override or add mapping in DEFAULT_TASK_MODEL_PATHS.'
        )
    return DEFAULT_TASK_MODEL_PATHS[normalized_task]


def resolve_importance_artifact_path(
    importance_root: Path,
    task_name: str,
    mode: str,
    tail: str,
    source_top_p_percent: float,
    prefer_exact_importance: bool,
    allow_exact_fallback: bool,
) -> tuple[Path, str]:
    """Resolve existing importance `.pt` path from config-driven naming pattern.

    Target filename pattern:
        `importance_{task_name}_{mode}_{resolved_tail}{top_p_label}.pt`

    Behavior:
        - Chooses base vs `_exact.pt` depending on preference and fallback flags.
        - Falls back to glob search when exact candidates are missing.

    Args:
        importance_root: Directory containing importance artifacts.
        task_name: Task key in filename.
        mode: Importance mode in filename.
        tail: Tail token (`top`, `bottom`, `auto`).
        source_top_p_percent: Source top-p percentage in filename.
        prefer_exact_importance: Whether to prefer `_exact.pt` first.
        allow_exact_fallback: Whether to allow `_exact.pt` as fallback.

    Returns:
        Tuple `(resolved_importance_path, resolved_tail)`.

    Raises:
        FileNotFoundError: If no matching artifact exists.
    """

    resolved_tail = resolve_importance_tail(mode=mode, tail=tail)
    mode_norm = mode.strip().lower()
    task_norm = task_name.strip().lower()
    top_p_label = format_percent_label(percent=source_top_p_percent)

    base_filename = f'importance_{task_norm}_{mode_norm}_{resolved_tail}{top_p_label}.pt'
    exact_filename = f'importance_{task_norm}_{mode_norm}_{resolved_tail}{top_p_label}_exact.pt'

    if prefer_exact_importance:
        candidate_filenames = [exact_filename, base_filename]
    else:
        candidate_filenames = [base_filename]
        if allow_exact_fallback:
            candidate_filenames.append(exact_filename)

    for filename in candidate_filenames:
        candidate_path = importance_root / filename
        if candidate_path.exists():
            return candidate_path, resolved_tail

    # If direct candidates are missing, perform a constrained glob fallback.
    glob_pattern = f'importance_{task_norm}_{mode_norm}_{resolved_tail}{top_p_label}*.pt'
    glob_matches = sorted(importance_root.glob(glob_pattern))

    if len(glob_matches) == 1:
        return glob_matches[0], resolved_tail

    if len(glob_matches) > 1:
        raise FileNotFoundError(
            'Multiple matching importance files were found; refine config to disambiguate. '
            f'pattern={glob_pattern}, matches={[str(path) for path in glob_matches]}'
        )

    raise FileNotFoundError(
        'No matching importance file found. '
        f'tried={candidate_filenames}, glob_pattern={glob_pattern}, root={importance_root}'
    )


def load_tokenizer_with_mistral_regex_fix(model_name_or_path: str) -> AutoTokenizer:
    """Load tokenizer with optional `fix_mistral_regex` compatibility argument.

    Args:
        model_name_or_path: Hugging Face model id or local checkpoint path.

    Returns:
        Loaded tokenizer instance.
    """

    try:
        return AutoTokenizer.from_pretrained(
            model_name_or_path,
            trust_remote_code=True,
            fix_mistral_regex=True,
        )
    except TypeError:
        return AutoTokenizer.from_pretrained(
            model_name_or_path,
            trust_remote_code=True,
        )


def load_causal_lm(
    model_name_or_path: str | Path,
    torch_dtype: torch.dtype,
    device: str,
) -> tuple[AutoModelForCausalLM, AutoTokenizer]:
    """Load CausalLM checkpoint and tokenizer on the target device.

    Args:
        model_name_or_path: Hugging Face model id or local checkpoint path.
        torch_dtype: Loading dtype.
        device: Runtime device string (typically `cpu` in this notebook).

    Returns:
        Tuple `(model, tokenizer)`.
    """

    resolved = str(model_name_or_path)
    model = AutoModelForCausalLM.from_pretrained(
        resolved,
        torch_dtype=torch_dtype,
        device_map=None,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    model.to(device)
    model.eval()

    tokenizer = load_tokenizer_with_mistral_regex_fix(resolved)
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer


def validate_parameter_compatibility(
    base_model: AutoModelForCausalLM,
    task_model: AutoModelForCausalLM,
) -> None:
    """Validate base/task parameter key and shape compatibility.

    Args:
        base_model: Base anchor model.
        task_model: Task model that provides dense task vector.

    Returns:
        None. Raises `ValueError` on incompatibility.
    """

    base_named = dict(base_model.named_parameters())
    task_named = dict(task_model.named_parameters())

    if set(base_named.keys()) != set(task_named.keys()):
        missing_in_task = sorted(set(base_named.keys()) - set(task_named.keys()))
        missing_in_base = sorted(set(task_named.keys()) - set(base_named.keys()))
        raise ValueError(
            'Parameter key mismatch between base and task models. '
            f'missing_in_task={missing_in_task[:5]}, missing_in_base={missing_in_base[:5]}'
        )

    for parameter_name, base_param in base_named.items():
        if tuple(base_param.shape) != tuple(task_named[parameter_name].shape):
            raise ValueError(
                f"Shape mismatch at '{parameter_name}': "
                f"base={tuple(base_param.shape)} vs task={tuple(task_named[parameter_name].shape)}"
            )


def load_importance_score_tensors(importance_path: Path) -> Dict[str, torch.Tensor]:
    """Load serialized importance mapping and normalize tensors for scoring.

    Args:
        importance_path: Path to serialized importance dictionary (`.pt`).

    Returns:
        Mapping `parameter_name -> importance_score_tensor` on CPU float32.

    Raises:
        FileNotFoundError: If artifact path is missing.
        TypeError: If loaded object is not a mapping of tensors.
    """

    if not importance_path.exists():
        raise FileNotFoundError(f'Importance artifact does not exist: {importance_path}')

    importance_obj = torch.load(importance_path, map_location='cpu')
    if not isinstance(importance_obj, Mapping):
        raise TypeError(f'Importance artifact must contain mapping, got {type(importance_obj)}')

    score_tensors: Dict[str, torch.Tensor] = {}
    for parameter_name, raw_tensor in importance_obj.items():
        if not torch.is_tensor(raw_tensor):
            raise TypeError(f"Importance entry '{parameter_name}' is not a tensor: {type(raw_tensor)}")

        # Importance should be non-negative, but we defensively apply absolute value
        # to tolerate tiny numeric sign noise in serialized artifacts.
        score_tensors[str(parameter_name)] = raw_tensor.detach().to(torch.float32).abs().cpu()

    return score_tensors


def validate_importance_score_compatibility(
    base_model: AutoModelForCausalLM,
    score_tensors: Mapping[str, torch.Tensor],
) -> None:
    """Validate importance score key/shape compatibility with base model 2D parameters.

    Only 2D+ parameters are checked because 1D parameters (normalization layers,
    biases) are not sparsified and do not require importance scores.

    Args:
        base_model: Base model providing authoritative parameter layout.
        score_tensors: Mapping loaded from importance artifact.

    Returns:
        None. Raises `ValueError` when key/shape mismatch is detected for 2D params.
    """

    for parameter_name, base_param in base_model.named_parameters():
        if not torch.is_floating_point(base_param.data):
            continue

        # Skip 1D parameters — they receive full delta without importance masking.
        if base_param.data.ndim < 2:
            continue

        if parameter_name not in score_tensors:
            raise ValueError(f'Missing importance score tensor for 2D parameter: {parameter_name}')

        if tuple(score_tensors[parameter_name].shape) != tuple(base_param.shape):
            raise ValueError(
                f"Importance score shape mismatch for '{parameter_name}': "
                f"score={tuple(score_tensors[parameter_name].shape)} vs model={tuple(base_param.shape)}"
            )


def is_sparse_candidate(param: torch.nn.Parameter) -> bool:
    """Check whether a parameter should undergo importance-based sparsification.

    Only 2D+ floating-point parameters (e.g., linear weight matrices in attention/MLP)
    are sparsified.  1D parameters (normalization scales/biases, embedding biases) are
    excluded — they always receive the full task-vector delta.

    Args:
        param: Model parameter to check.

    Returns:
        True if the parameter is a 2D+ floating-point tensor eligible for sparse masking.
    """

    return torch.is_floating_point(param.data) and param.data.ndim >= 2


def compute_layerwise_topk_mask(
    score: torch.Tensor,
    keep_ratio: float,
) -> torch.Tensor:
    """Compute exact top-k boolean mask for a single parameter's importance scores.

    Uses `torch.topk` on the flattened score tensor to select exactly
    `round(keep_ratio * numel)` coordinates.  Ties are broken deterministically
    by flattened index position (lower index wins).

    Args:
        score: Importance score tensor for one parameter (any shape, typically 2D).
        keep_ratio: Fraction of coordinates to keep, in `[0, 1]`.

    Returns:
        Boolean mask with the same shape as `score`, True for kept coordinates.

    Raises:
        ValueError: If `keep_ratio` is outside `[0, 1]`.
    """

    if keep_ratio < 0.0 or keep_ratio > 1.0:
        raise ValueError(f'keep_ratio must be in [0, 1], got {keep_ratio}')

    numel = score.numel()
    k = int(round(keep_ratio * numel))
    k = max(0, min(k, numel))

    # Edge cases: keep nothing or keep everything.
    if k == 0:
        return torch.zeros_like(score, dtype=torch.bool)
    if k >= numel:
        return torch.ones_like(score, dtype=torch.bool)

    # torch.topk returns the k largest values and their indices in the flattened
    # tensor.  sorted=False is faster; tie-breaking is by index (deterministic).
    flat_score = score.detach().to(torch.float32).reshape(-1)
    _, topk_indices = torch.topk(flat_score, k=k, largest=True, sorted=False)

    mask_flat = torch.zeros(numel, dtype=torch.bool)
    mask_flat[topk_indices] = True
    return mask_flat.reshape(score.shape)


def apply_sparse_task_update_layerwise_inplace(
    base_model: AutoModelForCausalLM,
    task_model: AutoModelForCausalLM,
    score_tensors: Mapping[str, torch.Tensor],
    keep_ratio: float,
) -> Dict[str, Any]:
    """Apply sparse task-vector update using **layer-wise** top-k masking on **2D parameters only**.

    Update rules:
        - 2D+ params: `theta = theta_base + mask_topk_layerwise(score) * (theta_task - theta_base)`
        - 1D params:  `theta = theta_task`  (full task-vector delta, no masking)

    This approach sparsifies each 2D weight matrix independently so that each
    layer keeps exactly `keep_ratio` fraction of its coordinates.  1D parameters
    (normalization scales, biases) are never sparsified — they always receive
    the full task-vector update, preserving normalization layer fidelity.

    Args:
        base_model: Base model overwritten in-place with sparse-updated weights.
        task_model: Task model that provides dense task-vector delta.
        score_tensors: Importance score mapping used for coordinate selection (2D params).
        keep_ratio: Requested per-layer keep ratio in `[0, 1]` for 2D params.

    Returns:
        Summary dictionary containing layer-wise and aggregate statistics.
    """

    validate_parameter_compatibility(base_model=base_model, task_model=task_model)

    base_named = dict(base_model.named_parameters())
    task_named = dict(task_model.named_parameters())

    # Aggregate counters.
    total_2d_elements = 0
    kept_2d_elements = 0
    count_2d_params = 0
    total_1d_elements = 0
    count_1d_params = 0

    # Per-layer statistics for 2D parameters (lightweight: name -> kept/total).
    per_layer_stats: Dict[str, Dict[str, Any]] = {}

    with torch.no_grad():
        for parameter_name, base_param in base_named.items():
            if not torch.is_floating_point(base_param.data):
                continue

            base_fp32 = base_param.data.detach().to(torch.float32)
            task_fp32 = task_named[parameter_name].data.detach().to(torch.float32)
            delta_task = task_fp32 - base_fp32

            if not is_sparse_candidate(base_param):
                # ── 1D parameter: apply full task-vector delta (no sparsification) ──
                merged_tensor = task_fp32  # equivalent to base + full delta
                total_1d_elements += int(base_param.numel())
                count_1d_params += 1
            else:
                # ── 2D+ parameter: apply layer-wise top-k sparse mask ──
                if parameter_name not in score_tensors:
                    raise ValueError(f'Missing importance score for 2D parameter: {parameter_name}')

                score = score_tensors[parameter_name]
                if tuple(score.shape) != tuple(base_param.shape):
                    raise ValueError(
                        f"Score shape mismatch for '{parameter_name}': "
                        f"score={tuple(score.shape)} vs model={tuple(base_param.shape)}"
                    )

                # Compute per-layer exact top-k binary mask.
                layer_mask = compute_layerwise_topk_mask(score=score, keep_ratio=keep_ratio)
                sparse_update = delta_task * layer_mask.to(torch.float32)
                merged_tensor = base_fp32 + sparse_update

                # Track per-layer statistics.
                layer_kept = int(layer_mask.sum().item())
                layer_total = int(layer_mask.numel())
                per_layer_stats[parameter_name] = {
                    'kept': layer_kept,
                    'total': layer_total,
                    'realized_ratio': float(layer_kept / max(layer_total, 1)),
                }

                total_2d_elements += layer_total
                kept_2d_elements += layer_kept
                count_2d_params += 1

            base_param.data.copy_(merged_tensor.to(base_param.dtype))

    return {
        'selection_mode': 'layerwise_topk_2d_only',
        'keep_ratio': float(keep_ratio),
        # 2D (sparsified) aggregate stats.
        'total_2d_elements': int(total_2d_elements),
        'kept_2d_elements': int(kept_2d_elements),
        'realized_2d_keep_ratio': float(kept_2d_elements / max(total_2d_elements, 1)),
        'sparse_2d_params': int(count_2d_params),
        # 1D (full-update) aggregate stats.
        'total_1d_elements': int(total_1d_elements),
        'fullupdate_1d_params': int(count_1d_params),
        # Per-layer detail (2D only).
        'per_layer': per_layer_stats,
    }


def save_sparse_checkpoint(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    output_dir: Path,
    metadata: Mapping[str, Any],
) -> None:
    """Save sparse-updated checkpoint and metadata.

    Args:
        model: Sparse-updated model instance.
        tokenizer: Tokenizer saved with model.
        output_dir: Checkpoint output directory.
        metadata: JSON-serializable metadata payload.

    Returns:
        None. Artifacts are written to disk.
    """

    output_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(output_dir, safe_serialization=True)
    tokenizer.save_pretrained(output_dir)
    save_json(metadata, output_dir / 'merge_metadata.json')

In [ ]:
# --------------------------------------------------------------------------------------
# Main execution: resolve configured importance artifact and build sparse checkpoints
# --------------------------------------------------------------------------------------

if not (0.0 < float(RUNTIME.importance_top_p_percent) <= 100.0):
    raise ValueError(
        f'importance_top_p_percent must be in (0, 100], got {RUNTIME.importance_top_p_percent}'
    )

for keep_percent in SPARSE_KEEP_PERCENTS:
    if keep_percent <= 0.0 or keep_percent > 100.0:
        raise ValueError(f'Each sparse keep percent must be in (0, 100], got {keep_percent}')

set_seed(RUNTIME.seed)
merge_dtype = resolve_torch_dtype(RUNTIME.model_dtype_name)

resolved_task_model_path = resolve_task_model_path(
    task_name=RUNTIME.task_name,
    task_model_path_override=RUNTIME.task_model_path_override,
)
resolved_importance_path, resolved_tail = resolve_importance_artifact_path(
    importance_root=RUNTIME.importance_root,
    task_name=RUNTIME.task_name,
    mode=RUNTIME.importance_mode,
    tail=RUNTIME.importance_tail,
    source_top_p_percent=RUNTIME.importance_top_p_percent,
    prefer_exact_importance=RUNTIME.prefer_exact_importance,
    allow_exact_fallback=RUNTIME.allow_exact_fallback,
)

if not resolved_task_model_path.exists():
    raise FileNotFoundError(f'Resolved task model path does not exist: {resolved_task_model_path}')

if not resolved_importance_path.exists():
    raise FileNotFoundError(f'Resolved importance path does not exist: {resolved_importance_path}')

importance_top_p_label = format_percent_label(RUNTIME.importance_top_p_percent)
run_namespace = (
    f"{RUNTIME.task_name.lower()}_"
    f"{RUNTIME.importance_mode.lower()}_"
    f"{resolved_tail}{importance_top_p_label}"
)
run_root = RUNTIME.output_root / run_namespace
checkpoint_root = run_root / 'checkpoints'
metadata_root = run_root / 'metadata'
summary_json_path = metadata_root / 'if_importance_sparse_layerwise_summary.json'
summary_csv_path = metadata_root / 'if_importance_sparse_layerwise_summary.csv'

checkpoint_root.mkdir(parents=True, exist_ok=True)
metadata_root.mkdir(parents=True, exist_ok=True)

print(f'Resolved task model path: {resolved_task_model_path}')
print(f'Resolved importance path: {resolved_importance_path}')
print(f'Run namespace: {run_namespace}')
print(f'Checkpoint root: {checkpoint_root}')
print(f'Selection mode: layerwise_topk_2d_only (1D params get full delta)')

# Load task model once because dense task-vector source is reused for all keep ratios.
task_model_for_delta, _ = load_causal_lm(
    model_name_or_path=resolved_task_model_path,
    torch_dtype=merge_dtype,
    device='cpu',
)

# Load one base model for compatibility checks and reusable tokenizer capture.
base_model_for_validation, base_tokenizer = load_causal_lm(
    model_name_or_path=RUNTIME.base_model_id,
    torch_dtype=merge_dtype,
    device='cpu',
)

validate_parameter_compatibility(
    base_model=base_model_for_validation,
    task_model=task_model_for_delta,
)

importance_scores = load_importance_score_tensors(importance_path=resolved_importance_path)
validate_importance_score_compatibility(
    base_model=base_model_for_validation,
    score_tensors=importance_scores,
)

# Report 1D vs 2D parameter split for transparency.
n_1d_params = sum(1 for _, p in base_model_for_validation.named_parameters()
                  if torch.is_floating_point(p.data) and p.data.ndim < 2)
n_2d_params = sum(1 for _, p in base_model_for_validation.named_parameters()
                  if torch.is_floating_point(p.data) and p.data.ndim >= 2)
print(f'Parameter split: {n_2d_params} sparse-eligible (2D+) | {n_1d_params} full-update (1D)')

del base_model_for_validation
gc.collect()

run_rows = []

for keep_percent in SPARSE_KEEP_PERCENTS:
    keep_ratio = float(keep_percent) / 100.0
    keep_tag = keep_percent_to_tag(keep_percent=keep_percent)
    output_dir = checkpoint_root / f'{RUNTIME.task_name.lower()}_delta_sparse_importance_{run_namespace}_{keep_tag}'

    if output_dir.exists() and not RUNTIME.overwrite_existing:
        print(f'[Skip] Output already exists and overwrite is disabled: {output_dir}')
        run_rows.append(
            {
                'keep_percent': float(keep_percent),
                'keep_ratio': float(keep_ratio),
                'status': 'skipped_existing',
                'output_dir': str(output_dir),
            }
        )
        continue

    # Each run loads a fresh base checkpoint so variants remain independent.
    sparse_model, _ = load_causal_lm(
        model_name_or_path=RUNTIME.base_model_id,
        torch_dtype=merge_dtype,
        device='cpu',
    )

    # Apply layer-wise top-k sparse update on 2D params; full delta on 1D params.
    sparse_summary = apply_sparse_task_update_layerwise_inplace(
        base_model=sparse_model,
        task_model=task_model_for_delta,
        score_tensors=importance_scores,
        keep_ratio=keep_ratio,
    )

    # Build metadata — per_layer detail is saved in JSON but excluded from CSV rows
    # to keep the summary table compact.
    metadata = {
        'created_at': now_iso(),
        'method': 'task_delta_sparse_by_importance_layerwise_topk_2d_only',
        'selection_mode': 'layerwise_topk_2d_only',
        'formula_2d': 'theta = theta_base + mask_topk_layerwise(importance) * (theta_task - theta_base)',
        'formula_1d': 'theta = theta_task  (full delta, no masking)',
        'score_definition': 'importance score from configured importance artifact',
        'base_model_id': str(RUNTIME.base_model_id),
        'task_name': str(RUNTIME.task_name),
        'task_model_path': str(resolved_task_model_path),
        'importance_path': str(resolved_importance_path),
        'importance_mode': str(RUNTIME.importance_mode),
        'importance_tail': str(resolved_tail),
        'importance_top_p_percent': float(RUNTIME.importance_top_p_percent),
        'keep_percent': float(keep_percent),
        'keep_ratio': float(keep_ratio),
        'sparse_summary': sparse_summary,
    }

    save_sparse_checkpoint(
        model=sparse_model,
        tokenizer=base_tokenizer,
        output_dir=output_dir,
        metadata=metadata,
    )

    run_rows.append(
        {
            'keep_percent': float(keep_percent),
            'keep_ratio': float(keep_ratio),
            'status': 'saved',
            'kept_2d_elements': int(sparse_summary['kept_2d_elements']),
            'total_2d_elements': int(sparse_summary['total_2d_elements']),
            'realized_2d_keep_ratio': float(sparse_summary['realized_2d_keep_ratio']),
            'total_1d_elements': int(sparse_summary['total_1d_elements']),
            'sparse_2d_params': int(sparse_summary['sparse_2d_params']),
            'fullupdate_1d_params': int(sparse_summary['fullupdate_1d_params']),
            'output_dir': str(output_dir),
        }
    )

    print(
        f"Saved sparse checkpoint | keep_percent={keep_percent:.3f}% "
        f"| 2D kept={sparse_summary['kept_2d_elements']:,}/{sparse_summary['total_2d_elements']:,} "
        f"(ratio={sparse_summary['realized_2d_keep_ratio']:.6f}) "
        f"| 1D full-update={sparse_summary['total_1d_elements']:,} "
        f"| path={output_dir}"
    )

    del sparse_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

summary_df = pd.DataFrame(run_rows)
display(summary_df)

summary_payload = {
    'created_at': now_iso(),
    'runtime': asdict(RUNTIME),
    'resolved': {
        'task_model_path': str(resolved_task_model_path),
        'importance_path': str(resolved_importance_path),
        'importance_tail': str(resolved_tail),
        'run_namespace': str(run_namespace),
        'checkpoint_root': str(checkpoint_root),
    },
    'method': 'task_delta_sparse_by_importance_layerwise_topk_2d_only',
    'selection_mode': 'layerwise_topk_2d_only',
    'sparse_keep_percents': [float(pct) for pct in SPARSE_KEEP_PERCENTS],
    'runs': run_rows,
}
save_json(summary_payload, summary_json_path)
summary_df.to_csv(summary_csv_path, index=False)

print(f'\nSaved run summary JSON: {summary_json_path}')
print(f'Saved run summary CSV: {summary_csv_path}')

# Final cleanup keeps long notebook sessions stable.
del task_model_for_delta
del importance_scores
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()